## Electrostatic problem
Electric potential and current density are produced in the coil.

In [ ]:
from ngsolve import *
from netgen.read_gmsh import ReadGmsh
from ngsolve.webgui import Draw
from netgen.csg import *
import math
import pyvista as pv
import numpy as np
from ngsolve.krylovspace import GMRes

mesh_path = '../../meshes/coil_box'
output_path = '../../output/case1/case1_ngsolve'

# Import geometries
mesh = ReadGmsh(mesh_path + ".msh")

for i in range(1, 3):
    # print(i)
    mesh.SetMaterial(i, f'{i}')

for i in range(1, 13):
    # print(i)
    mesh.SetBCName(i-1, f'{i}')

mesh = Mesh(mesh)

mesh.ngmesh.Save(mesh_path + ".vol")

In [3]:
mesh.ne, mesh.nv, mesh.GetMaterials(), mesh.GetBoundaries()

(69629,
 12090,
 ('1', '2'),
 ('1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12'))

In [4]:
# Define material, coil, and BC parameters

I_coil = 2191.9 # Input electric current [A]
sigma_coil = 62.83185 # Coil's electric conductivity [S/m]

In [5]:
sigma = {"1": 0.0, "2": sigma_coil}  # Electric conductivity [S/m]
sigma_cf = CoefficientFunction([sigma.get(mat, 0.0) for mat in mesh.GetMaterials()])
crosssection = Integrate(1, mesh, definedon=mesh.Boundaries("8"))

print(I_coil/(crosssection))
print (crosssection)

fespot = H1(mesh, order=1, definedon=mesh.Materials("2"), dirichlet="11")
phi,psi = fespot.TnT()
with TaskManager():
    bfa = BilinearForm(sigma_cf*grad(phi)*grad(psi)*dx).Assemble()
    inv = bfa.mat.Inverse(freedofs=fespot.FreeDofs(), inverse="sparsecholesky")
    lff = LinearForm(I_coil/crosssection*psi*ds("8")).Assemble()
    gfphi = GridFunction(fespot)
    gfphi.vec.data = inv * lff.vec

gfcurrdens = -sigma_cf*grad(gfphi)

30998147.073655326
7.0710678118656e-05


In [6]:
fespot_global = H1(mesh, order=1)
gfphi_global = GridFunction(fespot_global)
gfphi_global.Set(gfphi, definedon=mesh.Materials("2"))

fescurrden_global = VectorH1(mesh, order=3)
gfcurrdens_global = GridFunction(fescurrden_global)
gfcurrdens_global.Set(gfcurrdens, definedon=mesh.Materials("2"))



fes_j = VectorL2(mesh, order=0) # order 0 is sufficient for piecewise constant J
gfj = GridFunction(fes_j)
gfj.Set(gfcurrdens)

# transfer_cf = CF( [0.0 if mat == "1" else gfphi for mat in mesh.GetMaterials()] )
# gfphi_global.Set(transfer_cf)

In [7]:
# vtk = VTKOutput(mesh,coefs=[gfphi],names=["sol"],filename=output_path + "electric_potential",subdivision=0)
# vtk.Do()

res = pv.read(mesh_path + "_named" + ".msh")
points = res.points
gfphi_out = np.zeros((points.shape[0], 1))
gfcurrdens_out = np.zeros((points.shape[0], 3))

for i in range(points.shape[0]):
    point = mesh(points[i, 0], points[i, 1], points[i, 2])
    gfphi_out[i, :] = gfphi_global(point)
    gfcurrdens_out[i, :] = gfcurrdens_global(point)

res["electric_potential"] = gfphi_out
res["current_density"] = gfcurrdens_out

res.save(output_path + ".vtu")